# Badminton-Sense: EDA & Debugging

Exploratory notebook for understanding the data pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.download import load_config, parse_annotations
from pathlib import Path

config = load_config('../config/config.yaml')
print('Config loaded')

In [ ]:
# Load annotations
ann_dir = Path('../' + config['paths']['annotations']) / 'shuttleset_v1'
try:
    annotations = parse_annotations(ann_dir)
    print(f'Total strokes: {len(annotations)}')
    annotations.head()
except FileNotFoundError:
    print('Run 01_download_data.py first')

In [ ]:
# Class distribution
if 'type' in annotations.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    annotations['type'].value_counts().plot(kind='bar', ax=ax)
    ax.set_title('Stroke Type Distribution (ShuttleSet)')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# Inspect a processed pose sequence
processed_dir = Path('../' + config['paths']['processed'])
if (processed_dir / 'sequences.npy').exists():
    sequences = np.load(processed_dir / 'sequences.npy')
    labels = np.load(processed_dir / 'labels.npy')
    print(f'Sequences: {sequences.shape}')
    print(f'Labels: {labels.shape}, classes: {np.unique(labels)}')
    
    # Visualize one sequence
    idx = 0
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for d, name in enumerate(['X', 'Y', 'Z']):
        axes[d].imshow(sequences[idx, :, :, d].T, aspect='auto', cmap='viridis')
        axes[d].set_title(f'{name} coordinates')
        axes[d].set_xlabel('Frame')
        axes[d].set_ylabel('Keypoint')
    plt.suptitle(f'Sample sequence (class: {config["classes"]["names"][labels[idx]]})')
    plt.tight_layout()
    plt.show()
else:
    print('Run preprocessing pipeline first')